In [9]:
import requests
import json
import sys  
import os
import re
import pandas as pd

In [23]:
def is_valid_cas(cas_str):
    # Matches 2-7 digits, hyphen, 2 digits, hyphen, 1 check digit
    pattern = r'^\d{2,7}\d$'
    return bool(re.match(pattern, cas_str))

def cas_to_smiles(cas):

    if not is_valid_cas(cas):
        return "Invalid CAS number format"
    
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{cas}/property/IsomericSMILES/JSON"
    
    try:
        response = requests.get(url)
        response.raise_for_status() 
        data = response.json()
        
        smiles = data['PropertyTable']['Properties'][0]['IsomericSMILES']
        return smiles
    except requests.exceptions.RequestException as e:
        return f"Error fetching data: {e}"
    except (KeyError, IndexError):
        return "CAS number not found or SMILES not available"

In [24]:
df = pd.read_excel('data/envirotox_20260415204209.xlsx')

In [18]:
print(f"Initial number of rows: {len(df)}")

Initial number of rows: 2003


In [27]:
df['CAS'] = pd.to_numeric(df['CAS'], errors='coerce')
df = df.dropna(subset=['CAS'])
df['CAS'] = df['CAS'].astype(int).astype(str)
print(f"Number of rows after dropping invalid CAS: {len(df)}")

Number of rows after dropping invalid CAS: 1896


In [28]:
df['SMILES'] = df['CAS'].apply(cas_to_smiles)

In [32]:
df.head()

,CAS,Chemical name,Latin name,Trophic Level,Effect,Effect value,Unit,Test type,Test statistic,Duration,Duration (days),Effect is 5X above water solubility,Source,version,SMILES
0,50000,Formaldehyde;Formaldehyde,Morone saxatilis,FISH,Mortality,59.000,mg/L,A,LC50,96 hours,4,0,"Bills,T.D., L.L. Marking, and G.E. Howe, . Sen...",EnviroTox.v2,Error fetching data: 404 Client Error: PUGREST...
1,50066,"Phenobarbital;5-Ethyl-5-phenyl-1,3-diazinane-2...",Pimephales promelas,FISH,Mortality,485.000,mg/L,A,LC50,96 hours,4,0,"Russom, C.L., Bradbury, S.P., Broderius, S.J.,...",EnviroTox.v2,Error fetching data: 404 Client Error: PUGREST...
2,50215,Lactic acid;2-Hydroxypropanoic acid,Oreochromis mossambicus,FISH,Mortality,258.000,mg/L,A,LC50,96 hours,4,0,"Saha, N. C., Bhunia, F., & Kaviraj, A. (2006)....",EnviroTox.v2,Error fetching data: 404 Client Error: PUGREST...
3,50293,"p,p'-DDT;1,1'-(2,2,2-Trichloroethane-1,1-diyl)...",Perca sp,FISH,Mortality,0.015,mg/L,A,LC50,96 hours,4,0,"Bathe,R., K. Sachsse, L. Ullmann, W.D. Hormann...",EnviroTox.v2,Error fetching data: 404 Client Error: PUGREST...
4,50306,"2,6-Dichlorobenzoic acid;2,6-Dichlorobenzoic acid",Lepomis macrochirus,FISH,Mortality,120.000,mg/L,A,LC50,96 hours,4,0,"Mayer,F.L.,Jr., and M.R. Ellersieck. Manual of...",EnviroTox.v2,Error fetching data: 404 Client Error: PUGREST...


In [31]:
df.to_csv('data/chemicals_Imane.csv', index=False)